This notebook generates the geospatial files requiered for the SWOT-mission data analysis.


What has been done:

1. Read series and cats (all together)
2. Formatting dataframe: Assign unique ID to every HF based on fields 'hf_id', 'latitude', 'longitude', and remove SAFE HF (those with low probability of flooding, usually corresponding to hilly areas where SWOT data has large errors)
3. Expand parquet series from yearly to daily series (decomposition)
4. Saving outputs

***

__author__ = "Miguel González Jiménez, migj" <br>
__maintainer__ = "Miguel González Jiménez, migj"<br>
__email__ = "mgonzalez.j@gmv.com"

***

In [1]:
import SWOT as swot
import geopandas as gpd
import pandas as pd

from tqdm.notebook import tqdm

import json
import numpy as np
from datetime import datetime

### 1. Reading RAW VIIRS flood series

In [ ]:
all_in_one = r"..data/HF_Flood_Series_raw.csv"
df = pd.read_csv(all_in_one)

### 2. Formatting dataframe

In [ ]:
# A. From string to Python lists
for col in ['series_binary', 'series_pct_flood_pixels', 'series_distance_m']:
    df[col] = df[col].apply(json.loads)

tqdm.pandas(desc=f"Json loads")
df['dates'] = df['dates'].progress_apply(lambda x: [datetime.strptime(d, '%Y-%m-%d') for d in json.loads(x)])

# B. Removing NON-SAFE HFs in new_cat field (new HFs categorized)
df['cat_new'] = df['cat_new'].apply(lambda x: x.upper())
df = df.rename(dict(cat_new='zona_nombre'),axis=1)
df = df[~df['zona_nombre'].isin(['SAFE'])]

# C. Setting an ID to everty HFs grouped by lat, lon, and hf_id
df['ID'] = df.groupby(['hf_id', 'latitude', 'longitude'], sort=False).ngroup().add(1)
df["ID"] = df["ID"].astype(int).astype(str).str.zfill(len(str(100)))

### 3. Expand to Time-Series

In [ ]:
def _expand_series_to_df(df):
    """
    Expand per-facility annual JSON series into long-format daily DataFrame.
    Each row = one facility × one day.
    """
    records = []

    for _, row in tqdm(df.iterrows(), desc='Expanding series'):
        dates      = row['dates']
        occurrence = row['series_binary']
        pct        = row['series_pct_flood_pixels']
        dist       = row['series_distance_m']

        for i, date in enumerate(dates):
            records.append({
                'hf_id'        : row['hf_id'],
                'hf_payam'     : row['hf_payam'],
                'latitude'     : row['latitude'],
                'longitude'    : row['longitude'],
                'buffer_pixels': row['buffer_pixels'],
                'date'         : date,
                'occurrence'   : occurrence[i] if i < len(occurrence) else np.nan,
                'pct_flooded'  : pct[i] / 100   if i < len(pct)        else np.nan,
                'min_distance' : dist[i]         if i < len(dist)       else np.nan,
                'day_of_year'  : date.timetuple().tm_yday,
                'year'         : row['year'],
                'ID': row.get('ID', np.nan),
                'zona_nombre': row.get('zona_nombre', np.nan),
            })

    df_out = pd.DataFrame(records)

    # Impute distance for dry days: set to buffer maximum distance
    df_out.loc[(df_out['occurrence'] == 0) & (df_out['buffer_pixels'] == 40), 'min_distance'] = 2545.5
    df_out.loc[(df_out['occurrence'] == 0) & (df_out['buffer_pixels'] == 60), 'min_distance'] = 3818.0

    return df_out

df_series = _expand_series_to_df(df)

### 4. Saving

Saving static file METADATA


In [ ]:
retain_cols = ['hf_id', 'longitude', 'latitude', 'hf_payam', 'buffer_pixels', 'ID', 'zona_nombre']
STATIC_HF = df_series.groupby(['hf_id', 'longitude', 'latitude']).first().reset_index()[retain_cols]

STATIC_pathout = r"..data/HF_non-Safe_metadata.geojson"
gpd.GeoDataFrame(STATIC_HF, geometry=gpd.points_from_xy(x=STATIC_HF['longitude'], y=STATIC_HF['latitude'])
                 ).to_file(STATIC_pathout, driver='GeoJSON')

Saving final Time series csv file with records of HFs listed in metadata

In [ ]:
df_series_pathout = r"..data/HF_VIIRS_non-Safe_series.parquet"
df_series.to_parquet(df_series_pathout)